In [0]:
# ═══════════════════════════════════════════════
# 02_SILVER — Clean Bronze into analysis-ready Silver
# ═══════════════════════════════════════════════
from pyspark.sql.functions import col, round as spark_round, concat_ws, to_date

bronze = spark.table("bronze_card_spending")

silver = (
    bronze
    .filter(col("v4_1").isNotNull())                    # drop missing spend values
    .filter(col("Category") != "Aggregate")             # drop the aggregate total
    .withColumn(                                        # build a proper date (DD-MM format)
        "date",
        to_date(
            concat_ws("-",
                col("Time"),
                col("DayMonth").substr(4, 2),           # month
                col("DayMonth").substr(1, 2)            # day
            ),
            "yyyy-MM-dd"
        )
    )
    .select(                                            # keep only clean final columns
        "date",
        spark_round(col("v4_1"), 2).alias("spend_index"),
        col("Geography").alias("geography"),
        col("Category").alias("category")
    )
)

silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_card_spending")
print("Silver table written. Rows:", spark.table("silver_card_spending").count())

Silver table written. Rows: 4184
